# Smart Classroom Energy Forecasting Dashboard

This project predicts classroom electricity usage based on Wi-Fi occupancy data using an ARIMA time-series forecasting model.  
The dashboard visualizes real-time usage, predictions, and confidence intervals.

In [16]:
!pip install statsmodels plotly

In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.arima.model import ARIMA

In [18]:
hours = pd.date_range(start="2026-03-01", periods=200, freq="H")

occupancy = []

for h in hours:
    if 9 <= h.hour <= 16:   # class hours
        occupancy.append(np.random.randint(35,80))
    elif 17 <= h.hour <= 20:
        occupancy.append(np.random.randint(10,30))
    else:
        occupancy.append(np.random.randint(0,5))

df = pd.DataFrame({
    "timestamp": hours,
    "wifi_connections": occupancy
})

df.head()

/tmp/ipykernel_214/3634219565.py:1: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,timestamp,wifi_connections
0,2026-03-01 00:00:00,1
1,2026-03-01 01:00:00,3
2,2026-03-01 02:00:00,2
3,2026-03-01 03:00:00,4
4,2026-03-01 04:00:00,3


In [19]:
df["electricity_usage"] = df["wifi_connections"] * 2.8 + np.random.randint(5,15,len(df))

In [20]:
model = ARIMA(df['electricity_usage'], order=(2,1,2))
model_fit = model.fit()

In [21]:
forecast = model_fit.get_forecast(steps=12)

predicted = forecast.predicted_mean
confidence = forecast.conf_int()

In [22]:
fig = go.Figure()

# Actual electricity usage
fig.add_trace(go.Scatter(
    x=df['timestamp'],
    y=df['electricity_usage'],
    mode='lines',
    name='Actual Electricity Usage'
))

# Forecast
future_time = pd.date_range(df['timestamp'].iloc[-1], periods=12, freq="H")

fig.add_trace(go.Scatter(
    x=future_time,
    y=predicted,
    mode='lines',
    name='Predicted Usage'
))

# Confidence interval
fig.add_trace(go.Scatter(
    x=future_time,
    y=confidence.iloc[:,0],
    mode='lines',
    line=dict(dash='dash'),
    name='Lower Confidence'
))

fig.add_trace(go.Scatter(
    x=future_time,
    y=confidence.iloc[:,1],
    mode='lines',
    line=dict(dash='dash'),
    name='Upper Confidence'
))

fig.update_layout(
    title="Smart Classroom Electricity Forecast Dashboard",
    xaxis_title="Time",
    yaxis_title="Electricity Usage",
)

fig.show()

/tmp/ipykernel_214/2696035168.py:12: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [24]:
peak_threshold = 200

if predicted.max() > peak_threshold:
    print("⚠️ Warning: High electricity demand expected in upcoming hours")
else:
    print("Electricity demand within safe limits")

Electricity demand within safe limits
